# Granite 4.0 H-Tiny Fine-Tuning for Polish Tasks

**How to use this notebook:**
1. Go to [Google Colab](https://colab.research.google.com)
2. Click File → Upload notebook
3. Upload this file: `04-granite_tiny_polish_finetuning.ipynb`
4. Select Runtime → Change runtime type → GPU (T4)
5. Run all cells

This notebook demonstrates how to fine-tune IBM's Granite 4.0 H-Tiny model using LoRA for Polish language tasks.

**Requirements:**
- Google Colab with GPU (T4, V100, or A100)
- Training data in JSONL format
- ~2-4 hours for 2,000 examples on T4

**What you'll learn:**
1. Load and configure Granite Tiny with Unsloth
2. Prepare Polish training data
3. Configure LoRA for efficient fine-tuning
4. Train and evaluate the model
5. Save and deploy the fine-tuned model

## Step 1: Check GPU and Environment

In [ ]:
# Check GPU availability and type
!nvidia-smi

# Expected output:
# - Tesla T4 (15GB) - Free tier
# - Tesla V100 (16GB) - Pro tier
# - Tesla A100 (40GB) - Pro+ tier

In [ ]:
import torch
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB" if torch.cuda.is_available() else "N/A")

## Step 2: Install Dependencies

This will take 3-5 minutes.

In [ ]:
# Install Unsloth and dependencies with compatible versions
!pip install --upgrade --no-cache-dir --quiet unsloth
!pip install --quiet "transformers<=5.5.0" datasets trl accelerate bitsandbytes

print("✓ Installation complete!")
print("Note: You may see dependency warnings - these are usually safe to ignore.")

## Step 3: Upload Training Data

Your training data should be in JSONL format with this structure:

```json
{"instruction": "Polish instruction", "input": "Polish input text", "output": "Expected Polish output"}
```

Choose one of the upload methods below:

In [ ]:
# Option A: Upload from local machine
from google.colab import files

print("Please upload your polish_train.jsonl file:")
uploaded = files.upload()

# Verify upload
if 'polish_train.jsonl' in uploaded:
    print("✓ File uploaded successfully!")
    !wc -l polish_train.jsonl
else:
    print("⚠ Please upload a file named 'polish_train.jsonl'")

In [ ]:
# Option B: Load from Google Drive
from google.colab import drive

drive.mount('/content/drive')

# Update this path to your file location
# !cp /content/drive/MyDrive/path/to/polish_train.jsonl .

# Verify
# !wc -l polish_train.jsonl

In [ ]:
# Option C: Create sample data for testing
import json

sample_data = [
    {
        "instruction": "Wyodrębnij dane z tekstu i zwróć JSON.",
        "input": "Faktura nr FV/123/2026 z dnia 12.06.2026 na kwotę 4500 PLN dla firmy ABC Sp. z o.o.",
        "output": '{"typ":"faktura","numer":"FV/123/2026","data":"2026-06-12","kwota":"4500 PLN","firma":"ABC Sp. z o.o."}'
    },
    {
        "instruction": "Sklasyfikuj wiadomość.",
        "input": "Klient prosi o pilny kontakt w sprawie błędu produkcyjnego.",
        "output": "pilne / wymaga odpowiedzi"
    },
    {
        "instruction": "Podsumuj tekst po polsku w 3 punktach.",
        "input": "Projekt modernizacji infrastruktury IT obejmuje migrację do chmury, wdrożenie nowych systemów bezpieczeństwa oraz szkolenie pracowników. Budżet wynosi 500 tys. PLN, a termin realizacji to 6 miesięcy.",
        "output": "- Migracja do chmury\n- Wdrożenie systemów bezpieczeństwa\n- Szkolenie pracowników (budżet 500k PLN, 6 miesięcy)"
    }
]

# Save sample data
with open('polish_train.jsonl', 'w', encoding='utf-8') as f:
    for item in sample_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("✓ Sample data created (3 examples)")
print("⚠ This is just for testing - replace with your real data!")

In [ ]:
# Preview the data
!head -n 3 polish_train.jsonl

## Step 4: Load Model with Unsloth

This will take 2-3 minutes to download and load the model.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Configuration
max_seq_length = 2048  # Can increase to 4096 or 8192 if needed
dtype = None  # Auto-detect (Float16 for T4/V100, BFloat16 for A100)
load_in_4bit = True  # Use 4-bit quantization to save memory

print("Loading Granite 4.0 H-Tiny model...")
print("This may take 2-3 minutes...\n")

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/granite-4.0-h-tiny",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✓ Model loaded successfully!")
print(f"Model size: {model.get_memory_footprint() / 1e9:.2f} GB")
print(f"Max sequence length: {max_seq_length}")
print(f"Quantization: {'4-bit' if load_in_4bit else 'None'}")

## Step 5: Configure LoRA

LoRA (Low-Rank Adaptation) allows efficient fine-tuning by only training a small number of additional parameters.

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                      # LoRA rank (8, 16, 32, or 64)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=32,             # LoRA alpha (typically 2*r)
    lora_dropout=0.05,         # Dropout for regularization
    bias="none",               # Bias training strategy
    use_gradient_checkpointing="unsloth",  # Memory optimization
    random_state=42,
    use_rslora=False,          # Rank-stabilized LoRA
    loftq_config=None,         # LoftQ quantization
)

# Print trainable parameters
print("\n" + "="*50)
model.print_trainable_parameters()
print("="*50)
print("\n✓ LoRA configuration complete!")
print("Only ~0.5-1% of parameters will be trained.")

## Step 6: Prepare Dataset

Format the data using Granite's chat template.

In [ ]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("json", data_files="polish_train.jsonl", split="train")

print(f"Dataset loaded: {len(dataset)} examples")
print(f"\nSample example:")
print(dataset[0])

In [ ]:
# Format examples for Granite chat template
def format_example(example):
    """
    Format examples using Granite's chat template:
    <|start_of_role|>user<|end_of_role|>
    {instruction}
    {input}
    <|end_of_text|>
    <|start_of_role|>assistant<|end_of_role|>
    {output}
    <|end_of_text|>
    """
    instruction = example["instruction"]
    input_text = example.get("input", "")
    output = example["output"]
    
    # Combine instruction and input
    user_message = instruction
    if input_text:
        user_message += f"\n\n{input_text}"
    
    # Format with chat template
    text = f"""<|start_of_role|>user<|end_of_role|>
{user_message}
<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>
{output}
<|end_of_text|>"""
    
    return {"text": text}

# Apply formatting
print("Formatting dataset...")
dataset = dataset.map(format_example, remove_columns=dataset.column_names)

print("\n✓ Dataset formatted!")
print(f"\nFormatted example:")
print("="*50)
print(dataset[0]["text"])
print("="*50)

## Step 7: Configure Training

Set up training parameters optimized for T4 GPU.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# Training configuration
training_args = TrainingArguments(
    # Output
    output_dir="./granite4-tiny-h-polish-lora",
    
    # Batch size and accumulation
    per_device_train_batch_size=1,      # Batch size per GPU
    gradient_accumulation_steps=8,       # Effective batch size = 1 * 8 = 8
    
    # Learning rate and schedule
    learning_rate=2e-4,                  # LoRA typically uses 1e-4 to 5e-4
    lr_scheduler_type="cosine",          # Cosine decay
    warmup_steps=50,                     # Warmup steps
    
    # Training duration
    num_train_epochs=2,                  # 2-3 epochs typical for LoRA
    max_steps=-1,                        # -1 means use num_train_epochs
    
    # Optimization
    optim="adamw_8bit",                  # 8-bit AdamW saves memory
    weight_decay=0.01,                   # L2 regularization
    max_grad_norm=1.0,                   # Gradient clipping
    
    # Precision
    fp16=not torch.cuda.is_bf16_supported(),  # FP16 for T4/V100
    bf16=torch.cuda.is_bf16_supported(),      # BF16 for A100
    
    # Logging and saving
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,                  # Keep only 2 checkpoints
    
    # Other
    report_to="none",                    # Disable wandb/tensorboard
    seed=42,
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=training_args,
    packing=False,  # Don't pack multiple examples into one sequence
)

print("✓ Training configuration complete!")
print(f"\nTraining parameters:")
print(f"  Dataset size: {len(dataset)} examples")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Total steps: {len(dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs}")

## Step 8: Train the Model

**Estimated time:**
- 500 examples on T4: 30-60 minutes
- 2,000 examples on T4: 2-3 hours
- 5,000 examples on T4: 4-6 hours

**Note:** The cell will show progress bars and loss values during training.

In [ ]:
# Start training
print("Starting training...")
print("This may take a while. You can monitor progress below.\n")

trainer.train()

print("\n" + "="*50)
print("✓ Training complete!")
print("="*50)

## Step 9: Save the Model

Save the LoRA adapter and tokenizer.

In [ ]:
# Save LoRA adapter locally
model.save_pretrained("granite4-tiny-h-polish-lora")
tokenizer.save_pretrained("granite4-tiny-h-polish-lora")

print("✓ Model saved locally!")
print("\nSaved files:")
!ls -lh granite4-tiny-h-polish-lora/

In [ ]:
# Save to Google Drive (recommended)
from google.colab import drive

drive.mount('/content/drive')

!cp -r granite4-tiny-h-polish-lora /content/drive/MyDrive/

print("✓ Model saved to Google Drive!")
print("Location: /content/drive/MyDrive/granite4-tiny-h-polish-lora/")

## Step 10: Test the Fine-Tuned Model

Let's test the model with some Polish examples.

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

print("✓ Model ready for inference!")

In [ ]:
# Test function
def test_model(instruction, input_text="", max_new_tokens=512, temperature=0.7):
    """Test the fine-tuned model"""
    # Format prompt
    user_message = instruction
    if input_text:
        user_message += f"\n\n{input_text}"
    
    prompt = f"""<|start_of_role|>user<|end_of_role|>
{user_message}
<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>
"""
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    
    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Extract assistant response
    assistant_start = response.find("<|start_of_role|>assistant<|end_of_role|>")
    if assistant_start != -1:
        assistant_response = response[assistant_start + len("<|start_of_role|>assistant<|end_of_role|>"):].strip()
        # Remove end token if present
        assistant_response = assistant_response.replace("<|end_of_text|>", "").strip()
        return assistant_response
    
    return response

print("✓ Test function ready!")

In [ ]:
# Test 1: Document extraction
print("Test 1: Polish Document Extraction")
print("="*50)

result1 = test_model(
    "Wyodrębnij dane z tekstu i zwróć JSON.",
    "Faktura nr FV/456/2026 z dnia 15.06.2026 na kwotę 7800 PLN dla firmy XYZ Sp. z o.o."
)

print(result1)
print("\n")

In [ ]:
# Test 2: Classification
print("Test 2: Polish Message Classification")
print("="*50)

result2 = test_model(
    "Sklasyfikuj wiadomość.",
    "Klient dziękuje za szybką realizację zamówienia."
)

print(result2)
print("\n")

In [ ]:
# Test 3: Summarization
print("Test 3: Polish Summarization")
print("="*50)

result3 = test_model(
    "Podsumuj tekst po polsku w 3 punktach.",
    "Projekt modernizacji infrastruktury IT obejmuje migrację do chmury, wdrożenie nowych systemów bezpieczeństwa oraz szkolenie pracowników. Budżet wynosi 500 tys. PLN, a termin realizacji to 6 miesięcy."
)

print(result3)
print("\n")

In [ ]:
# Interactive testing
print("Interactive Testing")
print("="*50)
print("Enter your own Polish instructions to test the model.")
print("Leave blank to skip.\n")

instruction = input("Instruction (Polish): ")
if instruction:
    input_text = input("Input text (optional): ")
    
    print("\nGenerating response...\n")
    result = test_model(instruction, input_text)
    
    print("Response:")
    print("="*50)
    print(result)
    print("="*50)

## Step 11: Download Model (Optional)

Download the fine-tuned model to your local machine.

In [ ]:
# Create a zip file
!zip -r granite4-tiny-h-polish-lora.zip granite4-tiny-h-polish-lora/

# Download
from google.colab import files
files.download('granite4-tiny-h-polish-lora.zip')

print("✓ Model downloaded!")

## Next Steps

1. **Evaluate**: Test on a held-out test set
2. **Iterate**: Collect more data and retrain
3. **Deploy**: Use the model in production

### Deployment Options:

**Option A: Load adapter with base model**
```python
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/granite-4.0-h-tiny",
    max_seq_length=2048,
    load_in_4bit=True,
)
model.load_adapter("granite4-tiny-h-polish-lora")
```

**Option B: Merge and export**
```python
model = model.merge_and_unload()
model.save_pretrained("granite4-tiny-h-polish-merged")
```

### Resources:
- [IBM Granite Documentation](https://www.ibm.com/granite/docs/)
- [Unsloth Documentation](https://github.com/unslothai/unsloth)
- [Full Guide](GRANITE_TINY_FINETUNING_GUIDE.md)